# Static Features for Flood Forecasting: Focus on Slope / Relief / TRI / TPI
 
In this notebook, we start with **Slope / Relief** (terrain steepness and ruggedness).  

Other static features (can be addressed later) include:  
- **Drainage density (DD)** — density of the drainage network.  
  👉 Second most important; controls how fast and where water concentrates.  
- **Proximity to river / Floodplain area** — distance and area near rivers.  
  👉 Relevant for flood risk assessment near settlements.  
- **%Water (MSK)** — share of water surfaces (lakes, wide rivers).  
  👉 Has some impact, but less important for Bhutan due to limited lakes/floodplains.  

---

## Plan of Work / Steps

**Goal:**  
Compute terrain derivatives (Slope, Relief) from DEM data for each basin polygon (10 basins and 186 catchments).  
Append results to existing CSVs with basin statistics.

**Input data:**  
Located in `data/HydroSHEDS/`.  
These are HydroSHEDS rasters (DEM and derivatives) clipped to Bhutan + buffer (`Latitude 25.0°–29.5°, Longitude 87.0°–93.5°`) and reprojected to **EPSG:3857** for compatibility with polygon shapefiles.  

**Polygon boundaries:**  
- `data/boundaries/basins/Basin boundary.shp` (10 basins)  
- `data/boundaries/186_watershed/186 Watershed boundary.shp` (186 catchments)  

**Result:**  
Updated CSVs with new static features:  
- `data/boundaries/basins/basin_dem_acc_stats.csv`  
- `data/boundaries/186_watershed/watershed_dem_acc_stats.csv`  

**Steps:**  
1. **DEM preprocessing**  
   - Use HydroSHEDS DEM in EPSG:3857.  
   - Derive **slope and relief** (done in this notebook).  
   - TRI / TPI / curvature (planned for later).    

2. **Algorithms & libraries**  
   - **Slope / Relief:** сalculated with rioxarray + numpy (gradient filters) + rasterstats.  

3. **Zonal statistics per basin**  
   - Use `rasterstats.zonal_stats` to summarize each metric within basin polygons.  

4. **Save results**  
   - Append new columns to the two CSVs (`basin_dem_acc_stats.csv` and `watershed_dem_acc_stats.csv`).  
   - Column names example:  
     - `slope_mean`, `slope_p90`, `pct_slope_gt_15`, `relief`.  

---

In [27]:
from pathlib import Path
import os

# Speed up GeoPandas I/O via pyogrio (# pyogrio: fast I/O engine for GeoPandas (faster than fiona for large shapefiles)
os.environ["GEOPANDAS_IO_ENGINE"] = "pyogrio"

import numpy as np
import geopandas as gpd
import rioxarray as rxr  # rioxarray: xarray extension for raster data (GeoTIFF) with CRS/transform support
import pandas as pd
from rasterstats import zonal_stats # rasterstats.zonal_stats: compute per-polygon statistics from raster values 
from typing import Union             # typing.Union: type hint for arguments that can accept multiple types (e.g., Path OR GeoDataFrame)
import rasterio                      # rasterio: core library for reading/writing rasters (GeoTIFF), CRS handling, metadata, affine transforms
from rasterio.mask import mask       # rasterio.mask.mask: crop/mask raster arrays by polygon geometries (used to calculate % area above slope thresholds)
    

In [28]:
# Here you can insert paths to your own input rasters and output CSV/shapefiles.
# Adjust according to your project folder structure/needs.

# INPUTS (all in EPSG:3857)
dem_3857 = Path("../../data/HydroSHEDS/dem_Bhutan_and_buffer_EPSG3857.tif")

# Polygon boundaries (ID fields: ws_id for 186; basin for 10)
ws186 = Path("../../data/boundaries/186_watershed/186 Watershed boundary.shp")
bs10  = Path("../../data/boundaries/basins/Basin boundary.shp")

# CSVs to update (merge by ID, preserve row order)
csv186 = Path("../../data/boundaries/186_watershed/watershed_dem_acc_stats.csv")
csv10  = Path("../../data/boundaries/basins/basin_dem_acc_stats.csv")

In [29]:
# ====================================================
# What is "Slope" in terrain analysis?
# ====================================================
# - "Slope" = steepness of the terrain, i.e. the angle between the horizontal plane
#   and the terrain surface at each DEM cell.
#
# - Units:
#     * Degrees (0° = flat, 45° = rise equals run, 90° = vertical cliff).
#     * Sometimes expressed as percent rise (tan(angle) * 100).
#
# - How it is computed here:
#     1. From DEM (Digital Elevation Model), compute elevation gradients in X and Y
#        directions (∂z/∂x and ∂z/∂y) using central differences.
#     2. Combine them into overall gradient magnitude:
#            G = sqrt( (∂z/∂x)^2 + (∂z/∂y)^2 )
#     3. Convert to slope angle:
#            slope_rad = arctan(G)
#            slope_deg = slope_rad * (180 / π)
#
# - Interpretation:
#     * Low values (0–5°) → flat valleys or plains.
#     * Moderate values (10–30°) → rolling hills / slopes.
#     * High values (30–70°) → steep mountainous terrain.
#     * >70° → likely cliffs or artifacts.
#
# In this pipeline:
# - DEM resolution: ~90 m (3 arc-seconds HydroSHEDS)
# - Output slope is saved as GeoTIFF (float32, degrees, nodata = -9999).
# ====================================================




def build_slope_from_dem(dem_path: Path):
    """
    Build slope (degrees) from DEM (EPSG:3857) using central differences.
    Writes float32 GeoTIFF with numeric nodata (-9999.0).
    Returns:
      slope_tif (Path), dem_da (xr.DataArray), slope_deg (np.ndarray float32)
    """
    # Open DEM as 2D DataArray (masked)
    dem_da = rxr.open_rasterio(dem_path, masked=True).squeeze()  # shape: (rows, cols)

    # Pixel sizes (meters) from affine transform
    res_x = abs(dem_da.rio.transform()[0])
    res_y = abs(dem_da.rio.transform()[4])

    # Central-difference gradients (dz per meter in x/y)
    gy, gx = np.gradient(dem_da.values.astype("float64"), res_y, res_x)
    slope_rad = np.arctan(np.hypot(gx, gy))
    slope_deg = np.degrees(slope_rad).astype("float32")

    # Prepare DataArray for writing
    da_slope = dem_da.copy(data=slope_deg)

    # Remove conflicting CF metadata so rioxarray controls nodata/dtype
    da_slope.attrs.pop("_FillValue", None)
    da_slope.encoding.pop("_FillValue", None)
    da_slope.encoding.pop("dtype", None)

    # Use numeric nodata compatible with float32, replace NaNs before write
    slope_nodata = -9999.0
    da_slope = da_slope.where(np.isfinite(da_slope), other=slope_nodata)

    # Write GeoTIFF
    slope_tif = dem_path.parent / "dem_slope_deg_EPSG3857.tif"
    da_slope.rio.write_nodata(slope_nodata, inplace=True)
    da_slope.rio.to_raster(slope_tif, dtype="float32", compress="LZW")

    return slope_tif, dem_da, slope_deg


In [30]:
def dem_features_for_polys(
    shapefile: Union[Path, gpd.GeoDataFrame],   # path to polygons OR a ready GeoDataFrame
    id_field: str,                              # unique polygon id (e.g., "ws_id", "basin")
    dem_path: Path,                             # DEM raster (projected CRS; vertical units in meters)
    slope_tif: Path,                            # slope raster in degrees (float32)
    dem_nodata_fallback: float = -9999.0,       # fallback if DEM lacks nodata
    slope_nodata: float = -9999.0,              # nodata used in slope GeoTIFF
) -> pd.DataFrame:
    """
    Compute per-polygon Slope & Relief features:
      - slope_mean_deg
      - slope_p90_deg
      - pct_slope_gt_15 : % of polygon area with slope > 15°
      - pct_slope_gt_30 : % of polygon area with slope > 30°
      - relief_m = dem_max - dem_min

    Notes:
      * Polygons are reprojected to EPSG:3857 to match rasters.
      * We read only DEM min/max (to build relief), no other DEM stats here.
      * all_touched=False for conservative raster-polygon intersection.
    """
    # --- Load polygons; accept Path or GeoDataFrame; reproject to EPSG:3857 ---
    if isinstance(shapefile, gpd.GeoDataFrame):
        gdf = shapefile.to_crs(3857)
    else:
        if not shapefile.exists():
            raise FileNotFoundError(f"Shapefile not found: {shapefile}")
        if not dem_path.exists():
            raise FileNotFoundError(f"DEM not found: {dem_path}")
        if not slope_tif.exists():
            raise FileNotFoundError(f"Slope raster not found: {slope_tif}")
        gdf = gpd.read_file(shapefile, engine="pyogrio").to_crs(3857)

    if id_field not in gdf.columns:
        raise KeyError(f"ID field '{id_field}' not found in shapefile columns: {gdf.columns.tolist()}")

    # --- DEM nodata (resolve from file; fallback if missing) ---
    dem_da = rxr.open_rasterio(dem_path, masked=True).squeeze()
    dem_nodata = dem_da.rio.nodata if dem_da.rio.nodata is not None else dem_nodata_fallback

    # ---- Relief needs only DEM min/max per polygon
    # all_touched=False → conservative: count only pixels whose center is inside polygon
    # all_touched=False → count only pixels whose center lies inside polygon (conservative)
    # all_touched=True  → include any pixel touched by polygon boundary (liberal)
    # Relief = dem_max - dem_min (meters)
    dem_minmax = zonal_stats(
        gdf, dem_path,
        stats=["min", "max"],
        nodata=dem_nodata,
        all_touched=False,
    )
    df_dem_minmax = pd.DataFrame(dem_minmax)
    relief_m = (df_dem_minmax["max"] - df_dem_minmax["min"]).rename("relief_m")

    # ---- Slope statistics per polygon (degrees)
    slope_mean = zonal_stats(
        gdf, slope_tif,
        stats=["mean"],
        nodata=slope_nodata,
        all_touched=False,
    )
    df_slp_mean = pd.DataFrame(slope_mean).rename(columns={"mean": "slope_mean_deg"})

    # p90 using legacy key supported by this rasterstats version
    slope_p90 = zonal_stats(
        gdf, slope_tif,
        stats=["percentile_90"],
        nodata=slope_nodata,
        all_touched=False,
    )
    df_slp_p90 = pd.DataFrame(slope_p90).rename(columns={"percentile_90": "slope_p90_deg"})

    # ---- % area with slope > 15° and > 30°
    
    def _pct_gt_for_geoms(slope_path: Path, geoms, nodata_val: float, thr: float) -> pd.Series:
        vals = []
        with rasterio.open(slope_path) as src:
            for geom in geoms:
                arr, _ = mask(src, [geom], crop=True, filled=True, nodata=nodata_val)
                a = arr[0].astype("float32")
                m = (a != nodata_val) & np.isfinite(a)
                vals.append(0.0 if m.sum() == 0 else 100.0 * (a[m] > thr).mean())
        return pd.Series(vals)

    pct15 = _pct_gt_for_geoms(slope_tif, gdf.geometry, slope_nodata, 15.0).rename("pct_slope_gt_15")
    pct30 = _pct_gt_for_geoms(slope_tif, gdf.geometry, slope_nodata, 30.0).rename("pct_slope_gt_30")

    # ---- Combine outputs and attach ID first
    df = pd.concat([df_slp_mean, df_slp_p90, pct15, pct30, relief_m], axis=1)
    out = pd.concat([gdf[[id_field]].reset_index(drop=True), df], axis=1)

    # Optional column order
    out = out[[id_field, "slope_mean_deg", "slope_p90_deg", "pct_slope_gt_15", "pct_slope_gt_30", "relief_m"]]
    return out

In [31]:
def merge_into_csv(new_df: pd.DataFrame, csv_path: Path, id_field: str):
    """
    Update existing CSV by ID: add missing columns, overwrite existing,
    preserve original row order (if CSV exists).
    """
    feature_cols = [c for c in new_df.columns if c != id_field]

    if csv_path.exists():
        old = pd.read_csv(csv_path)
        merged = old.merge(
            new_df[[id_field] + feature_cols],
            on=id_field,
            how="left",
            suffixes=("", "_new"),
        )
        # Overwrite existing columns with *_new where present
        for c in feature_cols:
            newc = f"{c}_new"
            if newc in merged.columns:
                merged[c] = merged[newc].where(~merged[newc].isna(), merged.get(c))
                merged.drop(columns=[newc], inplace=True)
        merged.to_csv(csv_path, index=False)
        print(f"Updated (merged): {csv_path} | rows={len(merged)} | added/overwritten cols={len(feature_cols)}")
    else:
        new_df.to_csv(csv_path, index=False)
        print(f"Created: {csv_path} | rows={len(new_df)} | cols={len(new_df.columns)}")

In [32]:
# Build slope once; keep DEM DataArray and slope array in memory for speed
slope_tif, dem_da, slope_deg_arr = build_slope_from_dem(dem_3857)

# 186 catchments
df186 = dem_features_for_polys(
    shapefile=ws186,
    id_field="ws_id",
    dem_path=dem_3857,
    slope_tif=slope_tif,
)
merge_into_csv(df186, csv186, id_field="ws_id")

# 10 large basins
df10 = dem_features_for_polys(
    shapefile=bs10,
    id_field="basin",
    dem_path=dem_3857,
    slope_tif=slope_tif,
)
merge_into_csv(df10, csv10, id_field="basin")

Updated (merged): ../../data/boundaries/186_watershed/watershed_dem_acc_stats.csv | rows=186 | added/overwritten cols=5
Updated (merged): ../../data/boundaries/basins/basin_dem_acc_stats.csv | rows=10 | added/overwritten cols=5


In [33]:
print("Columns (186):", pd.read_csv(csv186, nrows=1).columns.tolist())
print("Columns (10):",  pd.read_csv(csv10,  nrows=1).columns.tolist())

# Peek first rows to verify values landed in right IDs
display(pd.read_csv(csv186).head(3))
display(pd.read_csv(csv10).head(3))

Columns (186): ['ws_id', 'dem_min', 'dem_max', 'dem_mean', 'dem_std', 'dem_median', 'acc_min', 'acc_max', 'acc_mean', 'acc_std', 'acc_median', 'slope_mean_deg', 'slope_p90_deg', 'pct_slope_gt_15', 'pct_slope_gt_30', 'relief_m']
Columns (10): ['basin', 'dem_min', 'dem_max', 'dem_mean', 'dem_std', 'dem_median', 'acc_min', 'acc_max', 'acc_mean', 'acc_std', 'acc_median', 'slope_mean_deg', 'slope_p90_deg', 'pct_slope_gt_15', 'pct_slope_gt_30', 'relief_m']


,ws_id,dem_min,dem_max,dem_mean,dem_std,dem_median,acc_min,acc_max,acc_mean,acc_std,acc_median,slope_mean_deg,slope_p90_deg,pct_slope_gt_15,pct_slope_gt_30,relief_m
0,1,2690.0,4483.0,3871.240740,350.309780,3952.0,1.0,8601.0,77.982920,564.187823,3.0,20.347372,31.563908,71.168858,13.710903,1793.0
1,2,3433.0,6141.0,4576.269356,479.646825,4593.0,1.0,36739.0,113.261979,1023.329133,4.0,25.849602,39.229057,84.292095,34.745533,2708.0
2,3,3409.0,6447.0,4636.337642,468.489801,4651.0,1.0,36822.0,155.080198,1498.649756,4.0,24.806776,38.883635,80.012352,31.302660,3038.0


,basin,dem_min,dem_max,dem_mean,dem_std,dem_median,acc_min,acc_max,acc_mean,acc_std,acc_median,slope_mean_deg,slope_p90_deg,pct_slope_gt_15,pct_slope_gt_30,relief_m
0,Aiechhu,92.0,4160.0,1183.538707,699.525166,1091.0,1.0,3754703.0,210.078824,7825.328906,3.0,21.200437,34.513462,72.055599,21.698405,4068.0
1,Merak_Sakteng,2690.0,4483.0,3882.785851,340.278285,3965.0,1.0,10252.0,65.099067,487.918880,3.0,20.382105,31.582006,71.324777,13.740984,1793.0
2,Mangdechhu,109.0,7065.0,3230.588248,1302.878378,3287.0,1.0,979449.0,1070.502172,21801.824509,3.0,23.063430,35.971821,77.342031,25.226937,6956.0
